# MSGM: binary FACED, one subject-disjoint split
Train 10 epochs using log-PSD; select on validation and evaluate once on held-out test subjects.


In [ ]:
import os,sys,json,random,hashlib,time
from pathlib import Path
import numpy as np,torch,pandas as pd
from torch import nn
from sklearn.metrics import balanced_accuracy_score,accuracy_score
root=Path(r'D:\EEG Experiment Artifacts\2026-11-02');source=root/'mamba_sources'/'MSGM';sys.path.insert(0,str(source))
from msgm.model import MSGM,REGION_MAP_32_10
seed=42;random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed);torch.set_num_threads(4)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu');assert device.type=='cuda'
protocol=json.loads((root/'binary_de_psd'/'protocol.json').read_text(encoding='utf8'));split=protocol['splits'][0]
features=np.load(root/'binary_de_psd'/'cache'/'features.npy',mmap_mode='r');assert features.shape==(2952,30,30,10)
subjects=np.repeat(np.arange(123),24);labels=np.tile(np.r_[np.zeros(12,dtype=np.int64),np.ones(12,dtype=np.int64)],123)
assert not set(split['train'])&set(split['val']) and not set(split['train'])&set(split['test']) and not set(split['val'])&set(split['test'])
train_ids=np.flatnonzero(np.isin(subjects,split['train']))
train=np.asarray(features[train_ids,:,:,5:10],dtype=np.float32);mean=train.mean(axis=(0,1),keepdims=True);std=np.maximum(train.std(axis=(0,1),keepdims=True),1e-6);del train
region_ids=list(REGION_MAP_32_10[:16])+list(REGION_MAP_32_10[18:]);assert len(region_ids)==30
model=MSGM(num_channels=30,num_features=5,num_classes=2,scale_lengths=(30,),region_ids=region_ids).to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-3)
print('DEVICE',torch.cuda.get_device_name(0),'PARAMETERS',sum(p.numel() for p in model.parameters()),'train_subjects',len(split['train']),'val_subjects',len(split['val']),flush=True)


In [ ]:
# One 30-second scale uses existing one-second log-PSD rather than the paper's multi-scale rPSD.
# Each batch contains one subject only because the released model computes a graph from batch data.
def run(split_name,training):
 model.train(training);all_pred=[];all_y=[];losses=[]
 rng=np.random.default_rng(seed+(epoch if training else 0));subs=np.array(split[split_name]);
 if training:rng.shuffle(subs)
 for sub in subs:
  idx=np.flatnonzero(subjects==sub)
  if training:rng.shuffle(idx)
  for batch in np.array_split(idx,3):
   x=(np.asarray(features[batch,:,:,5:10],dtype=np.float32)-mean)/std;y=torch.from_numpy(labels[batch]).to(device);x=torch.from_numpy(x).to(device)
   with torch.set_grad_enabled(training):
    logits=model(x);loss=nn.functional.cross_entropy(logits,y)
    if training:optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
   losses.append(float(loss.detach().cpu()));all_pred.extend(logits.detach().argmax(1).cpu().numpy());all_y.extend(y.cpu().numpy())
 return {'loss':float(np.mean(losses)),'accuracy':float(accuracy_score(all_y,all_pred)),'balanced_accuracy':float(balanced_accuracy_score(all_y,all_pred))}


In [ ]:
output=root/'mamba_experiments'/'msgm_psd_single_split_10' ;output.mkdir(parents=True,exist_ok=True)
config={'model':'MSGM','input':'FACED official log PSD, 30 channels, five bands, thirty one-second windows','scales':[30],'graph':'author graph, batches within one subject','seed':seed,'fold':0,'epochs':10,'split':split,'source_commit':os.popen('git -C "'+str(source)+'" rev-parse HEAD').read().strip()}
(output/'config.json').write_text(json.dumps(config,indent=2),encoding='utf8');np.savez(output/'scaler.npz',mean=mean,std=std)
history=[];best=-1
for epoch in range(1,11):
 start=time.perf_counter();tr=run('train',True);va=run('val',False);row={'epoch':epoch,'train_loss':tr['loss'],'val_accuracy':va['accuracy'],'val_balanced_accuracy':va['balanced_accuracy'],'seconds':time.perf_counter()-start};history.append(row);pd.DataFrame(history).to_csv(output/'history.csv',index=False)
 if va['balanced_accuracy']>best:
  best=va['balanced_accuracy'];torch.save({'model':model.state_dict(),'epoch':epoch},output/'best_model.pt')
 print('EPOCH',epoch,json.dumps(row),flush=True)
checkpoint=torch.load(output/'best_model.pt',map_location=device,weights_only=True);model.load_state_dict(checkpoint['model']);test=run('test',False)
result={'best_validation_balanced_accuracy':best,'best_epoch':checkpoint['epoch'],'epochs':10,'test':test,'train_subjects':len(split['train']),'validation_subjects':len(split['val']),'test_subjects':len(split['test'])}
(output/'metrics.json').write_text(json.dumps(result,indent=2),encoding='utf8');print('COMPLETE',json.dumps(result),flush=True)
